## 3. Exercise: Observability with Langfuse

In this exercise, we will:
- Register a new **styled_visualizer** prompt in Langfuse Prompt Management.
- Create a new visualizer agent that uses this prompt.
- Build and run a graph that uses the new agent, wrapped in a Langfuse trace so you can inspect runs in the Langfuse UI.

In [ ]:
# ruff: noqa: E402
import sys
from pathlib import Path

# Construct src-path and append it to sys.path
src_path = Path.cwd().parent.parent
sys.path.append(str(src_path))

# Initialize settings-object that loads the OpenAI API, Tavily API keys and Langfuse credentials to the environment
from core.settings import settings  # noqa: F401


### 3.1 Register a styled_visualizer prompt in Langfuse

We first create a new Langfuse Prompt Management entry called `styled_visualizer`.
It follows the same **chat** structure as `visualizer_prompt` (system + user_plan roles) but enforces a more specific output formatting (e.g. sections and bullet points).

In [ ]:
import langfuse

from models.prompts import PromptRole

# Initialize Langfuse client (credentials come from settings imported above)
langfuse_client = langfuse.get_client()

styled_visualizer_system_prompt = '''
You are the Styled Chart Generator in a multi-agent system.
- When the user asks for a chart, you MUST call the `run_matplotlib` tool.
- The `code` argument must be valid Python that uses the provided `plt` and `ax`
  to fully define the plot. Do NOT include imports or figure creation.
- The `filename` argument should be a short, snake_case description ending with '.png',
  e.g. 'revenue_per_quarter.png'.
- After the tool returns, briefly describe the chart and INCLUDE the file path in your answer.
- Always design charts using the following visual style rules:
  - Background: white with light gray grid lines.
  - Use a consistent color palette:
    - primary series: '#1f77b4' (blue)
    - secondary series: '#ff7f0e' (orange)
    - tertiary/auxiliary series: '#2ca02c' (green)
  - Avoid more than 4 distinct colors; reuse the palette above.
  - For line charts: use solid lines with markers for important points.
  - For bar charts: use 80% opacity bars with thin, dark-gray edges.
  - For axes: use dark-gray ticks and labels; rotate x-axis labels if they overlap.
- At the very end of your message, output EXACTLY two lines:
  - FILE_PATH: <relative_path_to_chart_file>
  - CHART_NOTES: <one concise sentence summarizing the main insight in the chart>
  Do not include any other trailing text after these two lines.
'''

styled_visualizer_user_prompt = '''
**User question:** {user_query}

**Context:**

{context}
'''

langfuse_client.create_prompt(
    name="styled_visualizer",
    type="chat",
    prompt=[
        {"role": PromptRole.SYSTEM, "content": styled_visualizer_system_prompt},
        {"role": PromptRole.USER_PLAN, "content": styled_visualizer_user_prompt},
    ],
    labels=["dev_latest"],
)  # ty:ignore[no-matching-overload]


After running the cell above, open the **Langfuse UI** and verify that a prompt named `styled_visualizer` exists,
with type `chat`, label `dev_latest`, and the expected system/user prompt contents.

### 3.2 Point the existing visualizer to the new prompt

Instead of creating a new node, we reuse the existing `VisualizerNode` and visualizer agent from the main repository.
Those components obtain their prompts from the Prompt Registry via `settings.prompt_registry.visualizer`.

To make the newly created `styled_visualizer` prompt take effect, update your `settings.env` so that the visualizer prompt
configuration points to it, e.g.:

```env
prompt_registry__visualizer__name=styled_visualizer
prompt_registry__visualizer__type=chat
prompt_registry__visualizer__label=dev_latest
```

After changing `settings.env`, restart the notebook kernel so that `core.settings` picks up the new configuration.
From that point on, the existing visualizer agent and node will automatically use the `styled_visualizer` prompt
when the graph runs.

In [ ]:
# Optional: Inspect the current visualizer prompt registry configuration
from core.settings import settings

settings.prompt_registry.visualizer


### 3.3 Run the graph with a Langfuse trace

Finally, we define a runner that executes the existing multi-agent graph.
Because we pointed the visualizer prompt configuration in `settings.env` to `styled_visualizer`,
the existing `VisualizerNode` will now use the new styled prompt.
The runner is wrapped with a Langfuse trace (using `start_as_current_observation` and `propagate_attributes`)
similar to the CLI in `src/main.py`, so that the run appears in the Langfuse UI.

In [ ]:
from uuid import uuid4

from langfuse import get_client, propagate_attributes

from agents import PlannerNode
from graph import graph

from models.agents import Agents
from models.state import Message, State

traced_langfuse_client = get_client()

user_query = (
    "Create a barchart that informs about the distribution of quantities of rainfall in the different regions in Germany."
)
enabled_agents = [
    Agents.WEB_RESEARCHER,
    Agents.SYNTHESIZER,
    Agents.VISUALIZER,
    Agents.CHART_SUMMARIZER,
]

state = State(
    messages=[Message(content=user_query)],
    user_query=user_query,
    enabled_agents=enabled_agents,
)


In [ ]:
session_id = f"notebook-{uuid4()}"

with traced_langfuse_client.start_as_current_observation(
    as_type="span",
    name="styled-visualizer-notebook-run",
):
    with propagate_attributes(
        session_id=session_id,
        tags=["data-agent", "notebook", "styled-visualizer"],
        metadata={"query": user_query},
    ):
        await graph.run(start_node=PlannerNode(), state=state)

print(state.messages[-1].content)
